In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
  !git clone https://github.com/G-mikael/DL_PA1.git
  %cd /content/DL_PA1
  !git switch inferencia
  !pip install -r requirements.txt

In [ ]:
from pathlib import Path

# Obtém o caminho do diretório do notebook e sobe um nível para a raiz do projeto
ROOT_DIR = Path().resolve()
CHECKPOINT_PATH = ROOT_DIR / "checkpoints" / "trilha_a_unet_resnet18_seed2028.pth"

if str(ROOT_DIR) not in sys.path:
  sys.path.append(str(ROOT_DIR))

CHECKPOINT_PATH = ROOT_DIR / "checkpoints" / "trilha_a_best_iou.pth"
RAW_DIR = ROOT_DIR / "data" / "teste"


In [ ]:
import src.eval_single_image as eval_single_image

In [ ]:
import random

# Lista todos os subdiretórios em RAW_DIR e escolhe um aleatoriamente
samples = [d.name for d in RAW_DIR.iterdir() if d.is_dir()]
if not samples:
    raise FileNotFoundError(f"Nenhuma pasta de amostra encontrada em {RAW_DIR}")

In [ ]:
# Carrega o modelo
model = eval_single_image.load_model(CHECKPOINT_PATH)

sample_code = random.choice(samples)
print(f"Amostra selecionada aleatoriamente: {sample_code}")

# Executa
metrics = eval_single_image.process_single_image(sample_code, model, raw_dir=RAW_DIR)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
from pathlib import Path

caminho_imagem = "coloque/aqui/o/caminho/da/imagem.png"

img_path = Path(caminho_imagem)

# Fallback: Se o avaliador não preencher ou o arquivo não existir, usamos uma amostra padrão
if not img_path.is_file():
    print(f"⚠️ Atenção: Arquivo não encontrado em '{caminho_imagem}'.\nUsando amostra de teste padrão...")
    caminho_imagem = str(ROOT_DIR / "data/teste/00ae65c1c6631ae6f2be1a449902976e6eb8483bf6b0740d00530220832c6d3e/images/00ae65c1c6631ae6f2be1a449902976e6eb8483bf6b0740d00530220832c6d3e.png")
    img_path = Path(caminho_imagem) # ATUALIZA O PATH AQUI

print(f"\nProcessando imagem: {img_path.name}")

# Chama a função de predição (que vamos criar no passo 2)
mask_instancias = eval_single_image.predict_instance_mask(str(img_path), model)

# Calcula a contagem
instancias_unicas = np.unique(mask_instancias)
contagem = len(instancias_unicas) - 1 if 0 in instancias_unicas else len(instancias_unicas)

print(f"✅ Contagem final: {contagem} instâncias detectadas.")

# Visualização
img_original = cv2.imread(str(img_path))
img_original = cv2.cvtColor(img_original, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

axes[0].imshow(img_original)
axes[0].set_title("Imagem de Entrada", fontsize=14)
axes[0].axis("off")

mascara_plot = axes[1].imshow(mask_instancias, cmap="nipy_spectral", interpolation="nearest")
axes[1].set_title(f"Máscara de Instâncias (Contagem: {contagem})", fontsize=14)
axes[1].axis("off")

plt.tight_layout()
plt.show()